<a href="https://colab.research.google.com/github/obieshka/Python-2025-/blob/hw_6/%D0%BF%D1%80%D0%B0%D0%BA6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

# Загрузка данных
df = pd.read_csv('train.csv')

# Целевая переменная
X = df.drop(columns=['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'])
y = df['Survived']

# Определение числовых и категориальных признаков
numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# Предобработка пропусков
X['Age'].fillna(X['Age'].median(), inplace=True)
X['Fare'].fillna(X['Fare'].median(), inplace=True)
X['Embarked'].fillna(X['Embarked'].mode()[0], inplace=True)

# Разбиение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Конвейер предобработки
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

# Модель логистической регрессии
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Бейзлайн – константный классификатор
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)


"""
Выбор метрики для оценки качества модели

В задаче бинарной классификации (выжил / не выжил) целевой класс несбалансирован:
~38% выживших и ~62% погибших. В таких условиях accuracy может быть обманчива,
так как модель, предсказывающая всегда "не выжил", даст ~62% правильных ответов,
но не будет иметь практической ценности.

Основной метрикой выбираем F1-score — гармоническое среднее между precision и recall.
Это позволяет сбалансированно учитывать ошибки обоих типов:
- пропуск выживших (ложноотрицательные) и
- ложное предсказание выживания (ложноположительные).

Для полноты анализа дополнительно вычисляем accuracy, precision и recall,
а также (опционально) ROC-AUC, но основным критерием сравнения моделей
служит F1-score.
"""

# Оценка бейзлайна
print("=== Бейзлайн (константное предсказание) ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dummy):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dummy):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dummy):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_dummy):.4f}")

# Обучение модели логистической регрессии
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Оценка модели
print("\n=== Модель LogisticRegression ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.4f}")

=== Бейзлайн (константное предсказание) ===
Accuracy:  0.6145
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

=== Модель LogisticRegression ===
Accuracy:  0.8045
Precision: 0.7931
Recall:    0.6667
F1-score:  0.7244


/tmp/ipykernel_1674/3009710752.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X['Age'].fillna(X['Age'].median(), inplace=True)
/tmp/ipykernel_1674/3009710752.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr